# The Text Only Model

This model cannot be feed images or vision data, just text.

![text only model](../showcase_images/text-only-model.png)

In [ ]:
import EasyJupyter
import torch.nn as nn
import torch
from model.decoder import Decoder
from model.generator import Generator
from model.RoPE import precompute_freqs_cis
from typing import Optional

In [2]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig

In [ ]:
class TextOnlyModel(nn.Module):
    def __init__(self, cfg: BaseConfig):
        """
        The text-only model. This model cannot be feed images or audio, only text.
        """
        super().__init__()
        self.cfg = cfg
        self.decoder = Decoder(cfg).to(cfg.device)

        self.token_embedding = nn.Embedding(
            num_embeddings=cfg.vocab_size, embedding_dim=cfg.d_model
        )

        self.generator = Generator(cfg)

        # Precompute RoPE frequencies
        self.freqs_cis = None
        self.compute_RoPE()
        self.initialize_weights()

    def compute_RoPE(self):
        """
        Compute RoPE representations. Once at the initialization, then again whenever the size of the context length changes, e.g., after the initial stage of pre-training the context length is increased.
        """
        self.freqs_cis = precompute_freqs_cis(
            cfg=self.cfg,
            head_dim=self.cfg.d_model // self.cfg.attn_heads,
            theta=self.cfg.pos_embeddings_freq,
        ).to(self.cfg.device)

    def initialize_weights(self):
        """Initialize parameters with Xavier uniform"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

        print(
            f"\n\nModel initialized with {sum(p.numel() for p in self.parameters()):,} parameters!\n\n"
        )

    def forward(
        self,
        x,
        combined_mask: torch.Tensor,
        start_pos: int = 0,
        kv_cache: Optional[list[tuple]] = None,
    ):
        """
        Feed the model.
        Args:
            x: The input tensor of shape (batch_size, context_len).
            combined_mask: This mask contains the causal mask which prevents tokens from attending to future tokens, and the document mask which prevents a token in Doc A from attending to tokens in other documents.
                - Paper mentioned: "We use an attention mask that prevents self-attention between different documents within the same sequence. We find that this change had limited impact during in standard pre-training, but find it to be important in continued pre-training on very long sequences."
            start_pos: Handles the slicing of the documents for both training and inference. During training, start_pos is always 0, because we process the full context window at once. During inference, start_pos increments by 1 for each new token the model generates.
            kv_caches: Inference only, a list of kv caches.
        
        Returns:
            logits, updated_kv_cache
        """
        # Embed the input tokens
        x_embeds = self.token_embedding(x)

        # Slice the precomputed RoPE frequencies to match the context length, just in case the dataloader yields a smaller batch.
        context_len = x.shape[1]
        batch_freqs_cis = self.freqs_cis[start_pos:start_pos+context_len]

        # Run the Decoder
        decoder_out, updated_kv_cache = self.decoder(
            x_embeds, batch_freqs_cis, combined_mask, kv_cache
        )

        # Run the generator
        logits = self.generator(decoder_out)
        return logits, updated_kv_cache

In [5]:
# @i-c
# TEST
from llama_configs import Llama3_scaled_down
from model.utils.masking import create_causal_doc_mask

d_model = 512
vocab_size = 32000
context_len = 256
batch_size = 8
cfg = Llama3_scaled_down()

dummy_input = torch.randint(0, vocab_size, (batch_size, context_len), device=cfg.device)
mask = create_causal_doc_mask(cfg, dummy_input)


model = TextOnlyModel(cfg).to(cfg.device)
model(dummy_input, mask)


Project Root: /Users/tonyavis/Main/How_to_build_an_LLM


Model initialized with 99,528,704 parameters!




tensor([[[-0.1605,  0.3985, -0.1236,  ...,  0.1847, -0.0166,  0.0212],
         [-0.0715,  0.1982,  0.0079,  ...,  0.1320,  0.0071, -0.1069],
         [-0.1304,  0.1013, -0.0602,  ...,  0.2144,  0.0120, -0.1149],
         ...,
         [ 0.1277,  0.2826, -0.0025,  ...,  0.0202, -0.2076,  0.2997],
         [-0.0057,  0.2947, -0.0841,  ...,  0.1110, -0.2162,  0.4329],
         [ 0.0952,  0.2332, -0.0369,  ...,  0.0510, -0.1861,  0.3867]],

        [[ 0.0723,  0.2406,  0.1872,  ...,  0.0666,  0.0071,  0.1111],
         [-0.0124,  0.2542,  0.2576,  ...,  0.0660, -0.0558,  0.1481],
         [ 0.0971,  0.2841,  0.2703,  ...,  0.0642, -0.0819,  0.0676],
         ...,
         [ 0.0529,  0.5196,  0.5089,  ...,  0.3131,  0.0182,  0.1680],
         [-0.0409,  0.5081,  0.4250,  ...,  0.4663,  0.0286,  0.2338],
         [-0.0181,  0.5519,  0.4185,  ...,  0.4601,  0.0292,  0.1913]],

        [[ 0.1519, -0.0424, -0.3925,  ...,  0.2977,  0.1305,  0.1226],
         [-0.0424, -0.2060, -0.2675,  ...,  0